In [1]:
import numpy as np
import warnings
from solvers import gauss_seidel

print("=== KIỂM TRA PHƯƠNG PHÁP GAUSS-SEIDEL ===\n")

# ---------------------------------------------------------
# Test 1: Ma trận chéo trội (Diagonally Dominant) - Chắc chắn hội tụ
# ---------------------------------------------------------
print("Test 1: Diagonally Dominant")
A1 = [[4.0, 1.0, -1.0], 
      [2.0, 7.0, 1.0], 
      [1.0, -3.0, 12.0]]
b1 = [3.0, 19.0, 31.0]

x1 = gauss_seidel(A1, b1)
expected_x1 = np.linalg.solve(A1, b1)
if np.allclose(x1, expected_x1, atol=1e-5):
    print("✅ PASS: Đã giải đúng hệ chéo trội.\n")
else:
    print("❌ FAIL: Kết quả lệch.\n")

# ---------------------------------------------------------
# Test 2: Ma trận đối xứng xác định dương (SPD - Symmetric Positive Definite)
# ---------------------------------------------------------
print("Test 2: SPD (Symmetric Positive Definite)")
# Ma trận SPD không nhất thiết phải chéo trội nghiêm ngặt, nhưng Gauss-Seidel vẫn hội tụ.
A2 = [[2.0, -1.0, 0.0], 
      [-1.0, 2.0, -1.0], 
      [0.0, -1.0, 2.0]]
b2 = [1.0, 0.0, 1.0]

# Tắt cảnh báo chéo trội tạm thời để xem kết quả hội tụ
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    x2 = gauss_seidel(A2, b2)
    
expected_x2 = np.linalg.solve(A2, b2)
if np.allclose(x2, expected_x2, atol=1e-5):
    print("✅ PASS: Hội tụ thành công trên ma trận SPD.\n")
else:
    print("❌ FAIL: Kết quả lệch.\n")

# ---------------------------------------------------------
# Test 3: Không hội tụ (Non-convergent detect)
# ---------------------------------------------------------
print("Test 3: Non-convergent (Cảnh báo & Dừng)")
A3 = [[1.0, 3.0], 
      [4.0, 1.0]] # Không chéo trội
b3 = [4.0, 5.0]

try:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        x3 = gauss_seidel(A3, b3, max_iter=50) # Giới hạn vòng lặp nhỏ để test
        
        # Kiểm tra xem có bắt được cảnh báo không hội tụ không
        warning_msgs = [str(warn.message) for warn in w]
        if any("không hội tụ" in msg for msg in warning_msgs):
            print("✅ PASS: Đã phát hiện và cảnh báo thuật toán không hội tụ.\n")
        else:
            print("❌ FAIL: Thuật toán không hội tụ nhưng không cảnh báo.\n")
except Exception as e:
    print(f"❌ FAIL do lỗi bất ngờ: {e}\n")

# ---------------------------------------------------------
# Test 4: So sánh với Gauss (Comparison to Gauss/Numpy)
# ---------------------------------------------------------
print("Test 4: So sánh độ chính xác với Gauss (Numpy solver)")
A4 = [[10., -1., 2., 0.],
      [-1., 11., -1., 3.],
      [2., -1., 10., -1.],
      [0., 3., -1., 8.]]
b4 = [6., 25., -11., 15.]

x4_gs = gauss_seidel(A4, b4, tolerance=1e-8)
x4_gauss = np.linalg.solve(A4, b4)

diff = np.max(np.abs(np.array(x4_gs) - x4_gauss))
print(f"   Sai số lớn nhất giữa Gauss-Seidel và Gauss Elimination: {diff}")
if diff < 1e-6:
    print("✅ PASS: Độ chính xác tương đương Gauss Elimination.\n")
else:
    print("❌ FAIL: Sai số quá lớn.\n")

# ---------------------------------------------------------
# Test 5: Edge case - Số 0 trên đường chéo chính
# ---------------------------------------------------------
print("Test 5: Ngoại lệ - Số 0 trên đường chéo chính")
A5 = [[0.0, 2.0], 
      [3.0, 4.0]]
b5 = [1.0, 2.0]

try:
    gauss_seidel(A5, b5)
    print("❌ FAIL: Lẽ ra phải báo lỗi chia cho 0.\n")
except ValueError as e:
    print(f"✅ PASS: Đã chặn thành công -> {e}\n")

=== KIỂM TRA PHƯƠNG PHÁP GAUSS-SEIDEL ===

Test 1: Diagonally Dominant
✅ PASS: Đã giải đúng hệ chéo trội.

Test 2: SPD (Symmetric Positive Definite)
✅ PASS: Hội tụ thành công trên ma trận SPD.

Test 3: Non-convergent (Cảnh báo & Dừng)
✅ PASS: Đã phát hiện và cảnh báo thuật toán không hội tụ.

Test 4: So sánh độ chính xác với Gauss (Numpy solver)
   Sai số lớn nhất giữa Gauss-Seidel và Gauss Elimination: 1.4043033402799665e-10
✅ PASS: Độ chính xác tương đương Gauss Elimination.

Test 5: Ngoại lệ - Số 0 trên đường chéo chính
✅ PASS: Đã chặn thành công -> Phần tử trên đường chéo chính A[0][0] bằng 0, không thể chia được.

